# Stiffness 7 — consistent initialization

The previous notebook cut a gradient into a staircase, so every section boundary carried a jump. That mixed two costs: re-initializing the integrator, and resolving the discontinuity.

Here the elution of the three protein separation is split into sections that are **identical on both sides**. The inlet does not change across a boundary, the physics does not change, the solution does not change. Only the number of boundaries does.

Whatever that costs is the transition itself.

Three knobs, because there are three questions:

| knob | question |
|---|---|
| `n_sections` | what does one boundary cost? |
| `binding` | does it cost more for a **DAE** than for an **ODE**? |
| `init_step_size` | can it be bought back? |

The middle one is the real subject. Rapid equilibrium turns the bound states into *algebraic* variables, so CADET has to solve for them at every boundary rather than carry them across. That ought to make a DAE more expensive to restart than an ODE.

In [ ]:
import utility.initialization_explorer as explorer

explorer.show_setting()

## What to watch

The record reports **extra steps per boundary** against the same run done in one piece, so it is directly the price of a transition.

Watch the **mean BDF order** especially. Everywhere else in this chapter it has refused to leave 5.00; here it is the quantity that moves most.

In [ ]:
from ipywidgets import interact

show_init = explorer.make_explorer()

interact(show_init, **show_init.sliders);

## What happened

**A boundary that changes nothing still costs about fifty steps.**

| boundaries | steps | per boundary | mean BDF order |
|---|---|---|---|
| 0 | 1661 | — | **5.00** |
| 9 | 2084 | 47.0 | 4.41 |
| 24 | 2843 | 49.2 | 3.77 |
| 49 | 4010 | 47.9 | **3.21** |

Nothing is discontinuous. The same chromatogram is being computed either way. Forty-nine invisible boundaries cost **2349 extra steps** and drag the average order from 5 down to 3.2 — the integrator spends a large part of the run climbing back to the order it already had.

**And a DAE costs exactly what an ODE costs.**

| | ODE (kinetic) | DAE (rapid equilibrium) |
|---|---|---|
| 0 boundaries | 1661 | 1668 |
| 49 boundaries | 4010 | 4003 |
| per boundary | 47.9 | **47.7** |
| mean order | 3.207 | 3.204 |

Seven steps apart in four thousand. The prediction that algebraic constraints make re-initialization harder is simply wrong here: solving the bound states at a boundary costs nothing measurable. Whatever makes rapid equilibrium expensive in other settings, it is not the cost of initializing it.

**About half the bill is recoverable.** `INIT_STEP_SIZE` is the step the integrator restarts from, and at its default of 10⁻¹⁰ it has to double its way back up:

| `INIT_STEP_SIZE` | steps at 24 boundaries | mean order |
|---|---|---|
| 10⁻¹⁰ | 2843 | 3.77 |
| 10⁻⁴ | **2186** | 4.23 |

That is 27 of the 49 steps per boundary, bought back by one setting. The remaining 22 are the order rebuild, and no solver option recovers those.

`CONSISTENT_INIT_MODE` makes **no difference at all** — full and lean give identical step counts to the digit, for both the ODE and the DAE.

> **Take-away.** Section boundaries cost even when nothing happens at them, and the cost is the restart, not the discontinuity and not the algebra. Raising `INIT_STEP_SIZE` recovers about half of it; the rest is the price of starting again at order one. So the advice is the same as the previous notebook and now for a sharper reason: **do not create section boundaries you do not need.** A DAE is not the problem — an interruption is.